# BrownDye2 preparation: protein-ligand complex

0. Validate complex (chain IDs, SMILES)
1. Fix protein with PDBFixer, assign ligand topology with RDKit
2. Assemble complex PDB for tleap
3. AmberTools parameterization (antechamber, parmchk2, tleap) → prmtop/rst7
4. ParmEd convert to complex.pqr
5. APBS input generation using pdb2pqr inputgen
6. Run APBS

Steps 0-2 in this notebook, steps 3-6 via `scripts/browndye/run_amber_apbs.sh`.

In [ ]:
from pathlib import Path

from Bio.Data.IUPACData import protein_letters_3to1
from Bio.PDB import PDBIO, PDBParser, Select
from rdkit import Chem

from mdpp.prep.ligand import assign_topology
from mdpp.prep.protein import fix_pdb

# User configuration
COMPLEX_PDB = Path("complex.pdb")
WORKDIR = Path("tmp/")
WORKDIR.mkdir(exist_ok=True, parents=True)

LIGAND_SMILES = ""  # Required: canonical SMILES of the ligand
PROTEIN_CHAIN_ID = "A"
LIGAND_CHAIN_ID = "B"
PH = 7.4

## Step 0: Validate complex

In [ ]:
STANDARD_AA = {code.upper() for code in protein_letters_3to1}

parser = PDBParser(QUIET=True)
structure = parser.get_structure("complex", str(COMPLEX_PDB))
model = structure[0]

chains = {chain.id: chain for chain in model}
assert PROTEIN_CHAIN_ID in chains, f"Chain {PROTEIN_CHAIN_ID} not found. Available: {list(chains)}"
assert LIGAND_CHAIN_ID in chains, f"Chain {LIGAND_CHAIN_ID} not found. Available: {list(chains)}"

# Summarize selected chains
prot_chain = chains[PROTEIN_CHAIN_ID]
prot_resnames = {
    res.get_resname().strip()
    for res in prot_chain.get_residues()
    if res.get_resname().strip() not in ("HOH", "WAT")
}
n_prot_res = sum(1 for _ in prot_chain.get_residues())
non_standard = prot_resnames - STANDARD_AA
if non_standard:
    print(f"WARNING: chain {PROTEIN_CHAIN_ID} has non-standard residues: {non_standard}")
print(f"Protein chain {PROTEIN_CHAIN_ID}: {n_prot_res} residues ({len(prot_resnames)} unique)")

lig_chain = chains[LIGAND_CHAIN_ID]
lig_resnames = {res.get_resname().strip() for res in lig_chain.get_residues()}
n_lig_atoms = sum(1 for _ in lig_chain.get_atoms())
print(f"Ligand chain {LIGAND_CHAIN_ID}: residue(s) {lig_resnames}, {n_lig_atoms} atoms")

## Step 1a: Fix protein

In [ ]:
class _ChainSelect(Select):
    """Select all residues in a specific chain."""

    def __init__(self, chain_id: str) -> None:
        self.chain_id = chain_id

    def accept_chain(self, chain) -> int:
        return int(chain.id == self.chain_id)


pdb_io = PDBIO()
pdb_io.set_structure(structure)

# Extract protein chain
protein_pdb = WORKDIR / "protein.pdb"
pdb_io.save(str(protein_pdb), _ChainSelect(PROTEIN_CHAIN_ID))
print(f"Extracted protein chain {PROTEIN_CHAIN_ID} -> {protein_pdb}")

# Fix protein (add missing residues, atoms, hydrogens)
protein_fixed_pdb = WORKDIR / "protein_fixed.pdb"
fix_pdb(protein_pdb, protein_fixed_pdb, pH=PH)
print(f"Fixed protein -> {protein_fixed_pdb}")

## Step 1b: Assign ligand topology

In [ ]:
# Extract ligand chain
ligand_pdb = WORKDIR / "ligand.pdb"
pdb_io.save(str(ligand_pdb), _ChainSelect(LIGAND_CHAIN_ID))
print(f"Extracted ligand chain {LIGAND_CHAIN_ID} -> {ligand_pdb}")

# Validate SMILES and compute net charge
template_mol = Chem.MolFromSmiles(LIGAND_SMILES, sanitize=True)
assert template_mol is not None, f"Invalid SMILES: {LIGAND_SMILES}"
ligand_net_charge = Chem.GetFormalCharge(template_mol)
print(f"SMILES: {Chem.MolToSmiles(template_mol)}")
print(f"Net charge: {ligand_net_charge}")

# Assign bond orders from SMILES template
pdb_mol = Chem.MolFromPDBFile(str(ligand_pdb), sanitize=True, removeHs=True)
assert pdb_mol is not None, f"RDKit failed to parse {ligand_pdb}"
assigned = assign_topology(pdb_mol, template_mol)

# Write SDF for antechamber
ligand_sdf = WORKDIR / "ligand.sdf"
w = Chem.SDWriter(str(ligand_sdf))
w.write(assigned)
w.close()
print(f"Wrote topology-assigned ligand -> {ligand_sdf}")

## Step 2: Assemble complex PDB for tleap

In [ ]:
RECORD_TYPES = {"ATOM", "HETATM", "TER"}


def _pdb_coord_lines(path: Path) -> list[str]:
    lines = [
        line
        for line in path.read_text().splitlines(keepends=True)
        if line[:6].strip() in RECORD_TYPES
    ]
    return lines


complex_pdb = WORKDIR / "complex.pdb"
with open(complex_pdb, "w") as f:
    for line in _pdb_coord_lines(protein_fixed_pdb):
        f.write(line)
    for line in _pdb_coord_lines(ligand_pdb):
        f.write(line)
    f.write("END\n")

print(f"Assembled complex -> {complex_pdb}")

## Steps 3-6: AmberTools, PQR, APBS

Edit the constants at the top of `scripts/browndye/run_amber_apbs.sh`, then run:

```bash
bash scripts/browndye/run_amber_apbs.sh
```

3. AmberTools parameterization (antechamber, parmchk2, tleap) → prmtop/rst7
4. ParmEd convert to complex.pqr
5. APBS input generation using pdb2pqr inputgen
6. Run APBS